# AlpasFarm: Train YOLOv8 on 4skwhnrscr Goat Anatomical & Health Dataset

This Google Colab notebook automates end-to-end training of **YOLOv8** on the **4skwhnrscr-2** goat dataset (2,991 images, 15,072 annotations across `goat_face`, `eye`, `mouth`, `ear`, and `goat_body`).

### Detected Anatomical & Health Targets:
- **Class 0 (`goat_face`)**: Facial landmark detection & head symmetry
- **Class 1 (`eye`)**: Ocular screening (FAMACHA conjunctival pallor, discharge, opacity)
- **Class 2 (`mouth`)**: Muzzle inspection (Contagious Ecthyma / Orf scabs, nasal discharge)
- **Class 3 (`ear`)**: Ear posture (drooping lethargy indicator, mange mite lesions)
- **Class 4 (`goat_body`)**: Full body posture, BCS (1-5), and bloat detection

### Step 1: Install Dependencies & Check GPU Acceleration

In [ ]:
!pip install -q ultralytics torchvision pillow matplotlib pyyaml
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository & Prepare Dataset

In [ ]:
import os, shutil
from pathlib import Path

# Clone AlpasFarm repository if not already cloned
if not Path('capstone').exists():
    !git clone https://github.com/Aberte-m4rlon/capstone.git

%cd /content/capstone
print("Working directory:", os.getcwd())

### Step 3: Dataset Configuration (data.yaml)

In [ ]:
data_yaml_path = Path('datasets/4skwhnrscr/yolo_dataset/data.yaml')
if not data_yaml_path.exists():
    # Create unified dataset from 4skwhnrscr raw archives if needed
    !python datasets/4skwhnrscr/prepare_yolo_dataset.py

with open(data_yaml_path, 'r') as f:
    print(f.read())

### Step 4: Train YOLOv8 on Cloud GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data=str(data_yaml_path.resolve()),
    epochs=50,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=15,
    save=True,
    plots=True,
    project='runs/colab_train',
    name='4skwhnrscr_goat_model',
    exist_ok=True
)

### Step 5: Evaluate Model Performance & Metrics

In [ ]:
from IPython.display import Image, display

# Validate on test split
metrics = model.val(data=str(data_yaml_path.resolve()), split='val')
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Display training curves and confusion matrix
results_img = Path('runs/colab_train/4skwhnrscr_goat_model/results.png')
if results_img.exists():
    display(Image(filename=str(results_img)))

### Step 6: Export & Download Trained Weights (best.pt / best.onnx)

In [ ]:
# Export model to ONNX format
onnx_path = model.export(format='onnx', imgsz=640)
print(f"ONNX Model exported at: {onnx_path}")

from google.colab import files
best_pt = Path('runs/colab_train/4skwhnrscr_goat_model/weights/best.pt')
if best_pt.exists():
    print("Downloading best.pt...")
    files.download(str(best_pt))

if Path(onnx_path).exists():
    print("Downloading best.onnx...")
    files.download(str(onnx_path))